# Request shapes and request boundaries — rerun on the B10 baseline

**Status:** a copy of `07_shapes.ipynb` run on the B10 baseline `1a872c09`
(31 cases, 176 executions, n = 8 for seeds and 4 for variants, 2026-09-25).
The original records how Q28 and Q29 were approached on the 29-case run
`42cab17a` at n = 1, and stays as that record. Q28 and Q29 have since been
ruled (D99, D100, D105, D106); this copy checks what the same analysis shows on
the run the study compares against. D107 records where it differs.

Two changes to the code, both forced by n > 1: an execution is keyed by case
**and** repetition (grouping by case alone pooled all repetitions into one
16-turn "request"), and the per-case tables show the labels over a case's
repetitions. The analysis is otherwise unchanged. In particular it still labels
a multi-turn execution with `interaction_label`, the rule D106 replaced with
the union; where that matters it is said below. No new run was made, and
nothing was written to the database.

**In short**

1. `shapes7` is stable across repetitions and metamorphic relations, but it
   **does** produce `other`: 25 of 176 executions (the D81 afternoon seed,
   the packing list's weather + climate, surf conditions' mixes; dinner too,
   under `interaction_label` only).
2. No rule based on the trace labels an advisory *case* `advisory`: all 32
   advisory executions consulted a reference tool first. Per request (D106) no
   execution is advisory; per turn, 19 no-tool closing turns are.
3. Splitting where the shape changes recovers **about 84 %** of boundaries in
   fused runs of 3, 5 and 8 requests. Merges still never mislabel.
4. The split error **is** now measurable: 19 of 176 executions are cut in two
   on their own (both D81 seeds, 8 each, and surf conditions, 3).

## 1. How this came up

The participant round labels every agent turn automatically, from the tools it
called. Two labels existed before this report:

| label | defined in | grain | problem for the participant analysis |
|---|---|---|---|
| `request_type` | D09 | 4 classes, one per function of §1.4 | too coarse to say *where* uncovered participant turns concentrate (§4.3 (3)) |
| coverage signature | D24 | exact tool sequence + buckets | too fine to carry a failure rate per class |

A **request shape** would sit between the two. The author's starting
hypothesis was to take the shapes clearly identifiable in the 29 cases:
enough classes to be informative, few enough to keep cells populated. The
test was to check that hypothesis against a full-suite `n = 1` run before
committing to it. The most recent verification run, `42cab17a`, is used
rather than a new one.

A shape is derived after the fact from `tool_call_trace` (`src/shapes.py`). No
application change and no event-model change are involved.

## 2. The candidate rules

Each rule is a pure function of the tool names one turn called. Reading the
trip (`get_trip_details`, `list_trip_blocks`, `get_itinerary`) is context and
ignored by every rule.

| rule | classes |
|---|---|
| `d09` | D09's precedence, re-derived: scheduling › venue_lookup › factual_qa › advisory |
| `shapes7` | advisory · factual.reference · factual.climate · factual.forecast · venue.search · scheduling.draft · scheduling.edit · **other** |
| `shapes9` | `shapes7` with scheduling.edit split into move / place / remove |
| `families` | the set of D09 families present, joined (Q08's option b) |

`shapes7` and `shapes9` take **no precedence**. A turn that does two things is
`other`, because a precedence would silently decide which half counts, and
the mixed turn is exactly what participants may do and the suite does not.
The one exception is a search that feeds a draft: that turn is the draft,
because it is the suite's own grounded-plan shape.

Three checks decide whether a rule is trusted on participants:

1. **Label vs. declaration.** Each execution lands on the label its case's
   declared tools imply.
2. **Metamorphic consistency.** A follow-up keeps its seed's label, unless
   its relation changes which tool is needed.
3. **Sizes and `other`.** No class is empty, and `other` is near zero on the
   suite.

In [1]:
import sys, pathlib, random, itertools, collections
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))

import pandas as pd

from src import survey, shapes
from src.cases import load_cases
from src.coverage import tools_required_by, tools_alternatively_required_by

pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 60)

# Pinned: the stored outputs describe this run and no other.
RUN_ID = "1a872c09-7070-4789-a806-904969f9185d"  # B10 baseline, 31 cases (copy of 07 on this run; prose updated where run-specific)
RULES = list(shapes.RULES)

print(survey.headline(RUN_ID))
print()
print(f"shape rule version {shapes.REQUEST_SHAPE_RULE_VERSION}; suite holds {len(load_cases())} cases")

run 1a872c09  started 2026-09-25T22:27:16
  notes         B10 baseline — overnight, started by tools/overnight_baseline.py
  agent         gemini-3.5-flash (released 2026-05-19)
  api           0.1.8 @ b954dbe9b545
  suite         a1d6a7f96d8d
  executions    176  (+0 superseded)
  failed        0
  scored        176, mean 9.26
  cost          $4.5164



shape rule version 0-candidates; suite holds 31 cases


## 3. Every turn, under every rule

One row per agent turn. `request_type` is the label the app stored (A03). The
`d09` column re-derives the same rule here, and the two must agree. Otherwise
the comparison below would be between two codebases, not two rules.

In [2]:
turns = survey.turn_frame(RUN_ID)
# n > 1: an execution is a case *and* a repetition; grouping by case alone would pool all n.
turns["execution"] = turns["case"] + "#" + turns["repetition"].astype(str)
for rule in RULES:
    turns[rule] = turns["tool_names"].map(shapes.RULES[rule])

disagree = turns[turns["d09"] != turns["request_type"]]
print(f"{len(turns)} turns; d09 re-derived vs stored request_type: {len(disagree)} disagreements")
turns[["execution", "turn", "tool_names", "request_type", "shapes7", "shapes9", "families"]]

214 turns; d09 re-derived vs stored request_type: 0 disagreements


,execution,turn,tool_names,request_type,shapes7,shapes9,families
0,advisory_packing_list#1,1,[get_region_climate],factual_qa,factual.climate,factual.climate,factual_qa
1,advisory_packing_list#2,1,[get_region_climate],factual_qa,factual.climate,factual.climate,factual_qa
2,advisory_packing_list#3,1,"[get_weather, get_region_climate]",factual_qa,other,other,factual_qa
3,advisory_packing_list#4,1,[get_region_climate],factual_qa,factual.climate,factual.climate,factual_qa
4,advisory_packing_list#5,1,"[get_weather, get_region_climate]",factual_qa,other,other,factual_qa
...,...,...,...,...,...,...,...
209,venue_lookup_sintra_lunch__dir1_coffee_not_lunch#3,1,[search_places],venue_lookup,venue.search,venue.search,venue_lookup
210,venue_lookup_sintra_lunch__dir2_lisbon_day#1,1,"[search_places, search_places]",venue_lookup,venue.search,venue.search,venue_lookup
211,venue_lookup_sintra_lunch__dir2_lisbon_day#3,1,"[search_places, search_places]",venue_lookup,venue.search,venue.search,venue_lookup
212,venue_lookup_sintra_lunch__dir2_lisbon_day#4,1,"[search_places, search_places, search_places]",venue_lookup,venue.search,venue.search,venue_lookup


## 4. Check 1 — label vs. declaration

Per execution, the interaction's label is compared with the label the case's
declared tools imply (`tools_required_by`, plus one member per `any_of`
group). The interaction's label ignores a closing turn that called nothing
(`shapes.interaction_label`).

In [3]:
cases = {c.case_id: c for c in load_cases()}

rows = []
for (case_id, repetition), frame in turns.groupby(["case", "repetition"]):
    case = cases[case_id]
    row = {"case": case_id, "repetition": repetition, "function": case.function, "relation": case.metamorphic_relation or "seed"}
    for rule in RULES:
        declared = shapes.declared_label(tools_required_by(case), tools_alternatively_required_by(case), rule)
        observed = shapes.interaction_label(list(frame[rule]))
        row[rule] = observed
        row[f"{rule} ok"] = observed == declared
        row[f"{rule} declared"] = declared
    rows.append(row)
execs = pd.DataFrame(rows)

print(f"{len(execs)} executions")
pd.DataFrame({rule: {"agree": int(execs[f"{rule} ok"].sum()), "of": len(execs)} for rule in RULES}).T

176 executions


,agree,of
d09,134,176
shapes7,133,176
shapes9,133,176
families,126,176


In [4]:
# Where the observed label and the declaration part ways, under any rule:
# per case, the declared label and the observed labels over its repetitions.
def observed(labels):
    return ", ".join(f"{k} {v}" for k, v in collections.Counter(labels).most_common())

off = execs[~execs[[f"{r} ok" for r in RULES]].all(axis=1)]["case"].unique()
pd.DataFrame([
    {"case": c, "n": int((execs["case"] == c).sum()),
     **{k: v for r in ["d09", "shapes7"] for k, v in
        ((f"{r} declared", execs.loc[execs["case"] == c, f"{r} declared"].iloc[0]),
         (r, observed(execs.loc[execs["case"] == c, r])))}}
    for c in off
])

,case,n,d09 declared,d09,shapes7 declared,shapes7
0,advisory_packing_list,8,advisory,factual_qa 8,advisory,"factual.climate 4, other 4"
1,advisory_packing_list__inv1_paraphrase,4,advisory,factual_qa 4,advisory,factual.climate 4
2,advisory_packing_list__inv3_irrelevant_detail,4,advisory,factual_qa 4,advisory,"other 2, factual.climate 2"
3,advisory_sintra_full_or_half_day,8,advisory,factual_qa 8,advisory,factual.reference 8
4,advisory_sintra_full_or_half_day__inv1_paraphrase,4,advisory,factual_qa 4,advisory,factual.reference 4
5,advisory_sintra_full_or_half_day__inv4_german,4,advisory,factual_qa 4,advisory,factual.reference 4
6,factual_qa_lisbon_surf_conditions,8,factual_qa,"factual_qa 6, other 2",factual.climate,"factual.climate 5, other 3"
7,scheduling_afternoon_then_tram,8,scheduling,scheduling 8,other,other 8
8,scheduling_dinner_from_offer,8,scheduling,other 8,scheduling.draft,other 8


**All four rules miss the 32 advisory executions**, every repetition of the
six advisory cases. For every advisory question the agent consulted a reference
tool before it answered (`get_region_climate` for the packing list, with
`get_weather` as well in some, which makes those `other`; `read_wikipedia`
for Sintra). The trace therefore never yields an advisory *case* on this suite
either.

Beyond them, `shapes7` misses **surf conditions** in 3 of 8 (a climate +
Wikipedia/search mix, `other`) and **dinner from offer** in 8 of 8. The dinner
miss is an artefact of `interaction_label`: search then draft across two turns
is `other` by turn labels, `scheduling.draft` by D106's union, which is the
declared label. Under the rule in force it agrees. The afternoon seed is
declared `other` (draft + move) and lands there in all 8.

## 5. Check 2 — metamorphic consistency

Each follow-up is compared with its seed. The table shows a single label where
the relation kept it, and `seed → variant` where it did not.

In [5]:
# Per case, the labels over its repetitions; a single label where seed and
# variant agree, and "seed → variant" where they do not.
by_case = execs.groupby("case")
meta = []
for case in cases.values():
    if not case.base_case_id or case.case_id not in by_case.groups:
        continue
    row = {"variant": case.case_id.split("__", 1)[1], "seed": case.base_case_id}
    for rule in RULES:
        a = observed(by_case.get_group(case.base_case_id)[rule])
        b = observed(by_case.get_group(case.case_id)[rule])
        row[rule] = b if a == b else f"{a} → {b}"
    meta.append(row)
pd.DataFrame(meta).sort_values(["seed", "variant"]).reset_index(drop=True)

,variant,seed,d09,shapes7,shapes9,families
0,inv1_paraphrase,advisory_packing_list,factual_qa 8 → factual_qa 4,"factual.climate 4, other 4 → factual.climate 4","factual.climate 4, other 4 → factual.climate 4",factual_qa 8 → factual_qa 4
1,inv3_irrelevant_detail,advisory_packing_list,factual_qa 8 → factual_qa 4,"factual.climate 4, other 4 → other 2, factual.climate 2","factual.climate 4, other 4 → other 2, factual.climate 2",factual_qa 8 → factual_qa 4
2,inv1_paraphrase,advisory_sintra_full_or_half_day,factual_qa 8 → factual_qa 4,factual.reference 8 → factual.reference 4,factual.reference 8 → factual.reference 4,factual_qa 8 → factual_qa 4
3,inv4_german,advisory_sintra_full_or_half_day,factual_qa 8 → factual_qa 4,factual.reference 8 → factual.reference 4,factual.reference 8 → factual.reference 4,factual_qa 8 → factual_qa 4
4,dir1_the_earthquake,factual_qa_alfama_history,factual_qa 8 → factual_qa 4,factual.reference 8 → factual.reference 4,factual.reference 8 → factual.reference 4,factual_qa 8 → factual_qa 4
5,inv1_paraphrase,factual_qa_alfama_history,factual_qa 8 → factual_qa 4,factual.reference 8 → factual.reference 4,factual.reference 8 → factual.reference 4,factual_qa 8 → factual_qa 4
6,dir1_july_instead,factual_qa_lisbon_november_weather,factual_qa 8 → factual_qa 4,factual.climate 8 → factual.climate 4,factual.climate 8 → factual.climate 4,factual_qa 8 → factual_qa 4
7,inv1_paraphrase,factual_qa_lisbon_november_weather,factual_qa 8 → factual_qa 4,factual.climate 8 → factual.climate 4,factual.climate 8 → factual.climate 4,factual_qa 8 → factual_qa 4
8,dir2_on_the_9th,scheduling_belem_morning,scheduling 8 → scheduling 4,scheduling.draft 8 → scheduling.draft 4,scheduling.draft 8 → scheduling.draft 4,scheduling+venue_lookup 8 → scheduling+venue_lookup 4
9,dir3_a_full_day,scheduling_belem_morning,scheduling 8 → scheduling 4,scheduling.draft 8 → scheduling.draft 4,scheduling.draft 8 → scheduling.draft 4,scheduling+venue_lookup 8 → scheduling+venue_lookup 4


Invariance relations keep their label, with one exception that is the seed's
own variability: the packing list is `other` in 4 of 8 (weather + climate),
its paraphrase in 0 of 4, its irrelevant-detail variant in 2 of 4. Whether the
agent adds a forecast to the climate lookup varies by repetition, not by
relation. As before, only `shapes9` lets a directional relation show:
`dir1_put_the_monastery_on` and `dir4_off_the_itinerary` change the tool the
request needs, and under `shapes9` they change label in every repetition.

## 6. Check 3 — class sizes and `other`

Executions per label. A class that a rule can produce but the suite never reaches is a class whose participant turns would all be uncovered.

In [6]:
producible = {
    "d09": ["scheduling", "venue_lookup", "factual_qa", "advisory"],
    "shapes7": ["advisory", "factual.reference", "factual.climate", "factual.forecast",
                "venue.search", "scheduling.draft", "scheduling.edit", "other"],
    "shapes9": ["advisory", "factual.reference", "factual.climate", "factual.forecast", "venue.search",
                "scheduling.draft", "scheduling.move", "scheduling.place", "scheduling.remove", "other"],
}
sizes = pd.DataFrame({rule: execs[rule].value_counts().reindex(classes, fill_value=0)
                      for rule, classes in producible.items()})
sizes.apply(lambda col: col.map(lambda v: "·" if pd.isna(v) else int(v)))

,d09,shapes7,shapes9
advisory,0,0,0
factual.climate,·,31,31
factual.forecast,·,16,16
factual.reference,·,32,32
factual_qa,86,·,·
other,·,25,25
scheduling,44,·,·
scheduling.draft,·,16,16
scheduling.edit,·,20,·
scheduling.move,·,·,12


**`other` is not empty on this run: 25 of 176 executions.** The afternoon seed
(8, draft + move), the packing list and its variant (6, weather + climate),
surf conditions (3), and dinner (8, by `interaction_label`; by D106's union
dinner is `scheduling.draft`, leaving 17). So a participant request that mixes
the same things can match a recorded signature. What `42cab17a` showed, that
the suite has no mixed turn, held for that run only (D107).

`shapes7` reaches six of its seven named classes, with 16 to 36 executions
each; `advisory` stays empty. At n = 8, `shapes9`'s place and remove classes
hold 4 executions each: still one variant apiece, so still too narrow to carry
a rate on their own.

**Where Q28 stands:** ruled. `shapes7`, `other` as its own row (D99);
`advisory` means no acting tool (D105); a multi-turn unit is labelled by the
union of its tools (D106).

## 7. The second problem: the participant round has no request boundary

The checks above have a boundary for free: in the simulation, one case is one
request. The participant round does not:

- **Turns are separated.** Each agent turn is one `study_suggestions` row,
  with `run_id`, `turn_index`, `user_turn` and its trace (D07, D08).
- **Requests are not.** The only grouping above a turn is the Tonic run
  (`agent.runs.ts`). It collects a member's turns until they have been quiet
  for 30 minutes, so one run can hold a weather question, a venue search
  and a move.

This breaks three things. D24's signature needs the number of member turns in
an interaction. Labelled per turn, `advisory` would fill up with "thanks" and
"yes, do it". And the two rounds would count different units.

Options considered (Q29):

| | a request is… | cost |
|---|---|---|
| A | one turn (D07 as written) | follow-up and acknowledgement turns inflate `advisory` |
| B | one Tonic run | several requests in a run become one `other` |
| **C** | a run split wherever the acting shape changes; turns that call nothing join the request before them | two same-shape requests in a row merge; one request that changes shape splits |
| — | the agent tags a new request | **ruled out**: changes the agent's prompt and output (root rule 4), and is one more output that can be wrong |
| — | an offline segmenter model over `user_turn` | admissible, since it runs after the fact like the judge, but adds a model to pin and validate |

C is a pure, versioned rule over fields already logged (`shapes.segment`).
The question was whether it works.

## 8. The simulation: fusing recorded executions

Recorded executions are chained into synthetic participant runs. Every
execution is exactly one Tonic run (checked below), so the original boundaries
are known and serve as ground truth. Unlike `42cab17a`, this run holds
multi-turn executions, so a fused run can fail in both directions: a missed
seam (merge) and a boundary inside an execution (split). The fusion happens in
memory, and the recorded rows are only read.

Two measurements:

- **All ordered pairs** of executions (176 × 175 = 30 800), exhaustively.
  Here "boundaries found" equals "split exactly", so a pair containing an
  execution that splits on its own counts as missed.
- **Random runs** of 3, 5 and 8 requests, 2 000 each, drawn with replacement
  (seed 28). "Boundaries found" counts true seams recovered, whatever else
  happened.

This is `shapes.segment`, shape only. Slash and reference, and the D100 rule
(shape + slash), are measured in `08_segmentation__baseline_1a872c09`.

In [7]:
# Precondition: execution = Tonic run, so an execution is a known request.
from src import db
from src.execution import latest_attempts
from src.survey import _rows

executions, _ = latest_attempts(_rows("study_case_executions", "run_id", RUN_ID))
ids = [i for row in executions for i in (row.get("suggestion_ids") or [])]
run_of = {r["id"]: r["run_id"] for r in
          db.admin().table("study_suggestions").select("id, run_id").in_("id", ids).execute().data}
spanning = [row["case_id"] for row in executions
            if len({run_of[i] for i in row.get("suggestion_ids") or []}) != 1]
print(f"{len(executions)} executions, {len(set(run_of.values()))} Tonic runs, "
      f"executions spanning more than one run: {spanning or 'none'}")

176 executions, 176 Tonic runs, executions spanning more than one run: none


In [8]:
requests = {e: list(f.sort_values("turn")["tool_names"]) for e, f in turns.groupby("execution")}
names = sorted(requests)

def fuse(seq, rule):
    """One synthetic run from recorded executions: (segment() result, true split)."""
    fn, labels, truth = shapes.RULES[rule], [], []
    for case in seq:
        start = len(labels)
        labels += [fn(tools) for tools in requests[case]]
        truth.append(list(range(start, len(labels))))
    return shapes.segment(labels), truth, labels

def boundaries(split):
    return {s[0] for s in split} - {0}

results = []
for rule in ["d09", "shapes7", "shapes9"]:
    pairs = list(itertools.permutations(names, 2))
    exact = sum(fuse(p, rule)[0] == fuse(p, rule)[1] for p in pairs)
    results.append({"rule": rule, "runs": "all pairs", "n": len(pairs),
                    "runs split exactly": exact / len(pairs), "boundaries found": exact / len(pairs)})
    rng = random.Random(28)
    for k in (3, 5, 8):
        n, exact, found, total = 2000, 0, 0, 0
        for _ in range(n):
            seg, truth, _ = fuse([rng.choice(names) for _ in range(k)], rule)
            exact += seg == truth
            found += len(boundaries(seg) & boundaries(truth))
            total += len(boundaries(truth))
        results.append({"rule": rule, "runs": f"{k} requests", "n": n,
                        "runs split exactly": exact / n, "boundaries found": found / total})
recovery = pd.DataFrame(results)
for col in ("runs split exactly", "boundaries found"):
    recovery[col] = recovery[col].map("{:.1%}".format)
recovery

,rule,runs,n,runs split exactly,boundaries found
0,d09,all pairs,30800,55.0%,55.0%
1,d09,3 requests,2000,35.1%,63.8%
2,d09,5 requests,2000,11.3%,62.5%
3,d09,8 requests,2000,3.3%,63.4%
4,shapes7,all pairs,30800,66.3%,66.3%
5,shapes7,3 requests,2000,48.9%,84.2%
6,shapes7,5 requests,2000,26.5%,83.8%
7,shapes7,8 requests,2000,12.6%,84.3%
8,shapes9,all pairs,30800,67.0%,67.0%
9,shapes9,3 requests,2000,50.1%,84.9%


### What goes wrong, and what it costs

A missed boundary could come from anywhere, so the failures are grouped by the
shapes of the two requests that met. A merge only matters if it mislabels
something, so every recovered request is also checked for a wrong label, and
the shape shares per run are compared with the truth.

In [9]:
shape_of = {c: shapes.interaction_label([shapes.shapes7(t) for t in requests[c]]) for c in names}

failed = collections.Counter()
for a, b in itertools.permutations(names, 2):
    seg, truth, _ = fuse((a, b), "shapes7")
    if seg != truth:
        failed[(shape_of[a], shape_of[b])] += 1
misses = pd.DataFrame([{"first": a, "second": b, "pairs missed": n} for (a, b), n in failed.most_common()])
print(f"shapes7: {sum(failed.values())} of {len(names) * (len(names) - 1)} pairs missed; "
      f"pairs of two different shapes among them: {sum(n for (a, b), n in failed.items() if a != b)}")
misses

shapes7: 10380 of 30800 pairs missed; pairs of two different shapes among them: 5738


,first,second,pairs missed
0,venue.search,venue.search,1260
1,factual.reference,factual.reference,992
2,factual.climate,factual.climate,930
3,other,venue.search,684
4,venue.search,other,684
5,factual.reference,other,608
6,other,factual.reference,608
7,other,other,600
8,factual.climate,other,589
9,other,factual.climate,589


In [10]:
rng = random.Random(28)
segments = mislabelled = 0
tv = []
for _ in range(2000):
    seq = [rng.choice(names) for _ in range(5)]
    seg, _, labels = fuse(seq, "shapes7")
    got = collections.Counter(shapes.interaction_label([labels[i] for i in s]) for s in seg)
    true = collections.Counter(shape_of[c] for c in seq)
    segments += len(seg)
    mislabelled += sum(1 for s in seg
                       if shapes.interaction_label([labels[i] for i in s]) not in {shape_of[c] for c in seq})
    g, t = sum(got.values()), sum(true.values())
    tv.append(0.5 * sum(abs(got[k] / g - true[k] / t) for k in set(got) | set(true)))
print(f"requests recovered: {segments}; carrying a shape absent from their run: {mislabelled}")
print(f"mean distance between recovered and true shape shares per run (total variation): {sum(tv) / len(tv):.3f}")

requests recovered: 9801; carrying a shape absent from their run: 892
mean distance between recovered and true shape shares per run (total variation): 0.157


### Reading the simulation

- **Seams are found about 84 % of the time** under `shapes7` (runs of 3, 5 and
  8), against about 63 % for `d09` and 85 % for `shapes9`: the same ordering
  as on `42cab17a`, at slightly higher levels.
- **The all-pairs figure (66 %) is lower because of splits, not merges.** Of
  the 10 380 pairs `shapes7` misses, 5 506 are missed only because one of the
  two executions splits on its own (dinner, afternoon, surf), 4 072 only
  because the seam merged, and 802 both. The "different shapes" count above
  is mostly the first kind: the split execution's `interaction_label` is
  `other`, whatever it met.
- **Merges still never mislabel.** The 892 of 9 801 recovered requests that
  carry a shape absent from their run are all halves of split executions,
  such as dinner's draft half, dinner's search half or surf's climate half.
  The pieces are named correctly; the whole was labelled `other` by
  `interaction_label`. No merge produced a wrong label.
- **Shape shares per run are closer to the truth** (total variation 0.157,
  against 0.217 on `42cab17a`), because at n = 8 same-case neighbours are
  common and merge into requests of the right shape.

## 9. What this simulation shows that `42cab17a` could not

The split error is now observable. 19 of 176 executions are cut in two by
shape alone: both D81 seeds in every repetition (dinner: search, then draft;
afternoon: draft, then a move) and surf conditions in 3 of 8 (a mixed first
turn, then climate). Whether reference repairs them is measured in
`08_segmentation__baseline_1a872c09`: it joins dinner and most of afternoon,
but neither a move linked only by wording nor surf's mixed opening.

What still cannot be shown is how people actually sequence requests. Random
draws ignore it, and real sessions probably have more same-shape neighbours
(a search, then "and coffee?"), whose status as a new request or a refinement
is ambiguous to a human reader too.

## 10. Where the decisions stand

| # | question | ruling | this run |
|---|---|---|---|
| Q28 | which rule labels a turn's shape | `shapes7`, `other` its own row (D99); `advisory` = no acting tool (D105); a multi-turn unit by the union (D106) | stable across repetitions and relations; `other` not empty (D107) |
| Q29 | how turns are grouped into requests | shape + slash, no reference (D100) | shape alone finds about 84 % of seams; 19 of 176 executions split on their own |

Both rules are versioned (`REQUEST_SHAPE_RULE_VERSION`, currently
`0-candidates`) and are frozen at C05, before any participant data exists.

The decision trail for how Q28 and Q29 were approached is in
`07_shapes.ipynb` §10 and is not repeated here.